## Using STED-FM model in PhenoMe pipeline

### 0. Setup

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from stedfm import get_pretrained_model_v2
from phenome.utils import ModelWrapper
from phenome import PhenoMe, load_dinov2_model

# Initialize PhenoMe object
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

#### Choose dataset and model to use

**NAS:** 224x224 crops of STED images of PSD-95 under 4 experimental conditions: 0MgGlyBic, 48hTTX, Block, and GluGly.

**Optim:** 224x224 crops of STED images of 4 different proteins: CaMKII, PSD-95, Actin and Tubulin.

**Zooniverse:** Crops of individual synaptic protein clusters. This is the only dataset with masks appropriate for extracting properties.


In [2]:
# Choose one of 3 datasets from ("NAS", "Optim", "Zooniverse")
#DATASET_NAME = "NAS"
#DATASET_NAME = "Optim"
DATASET_NAME = "Zooniverse"

# Choose one of 2 models from ("STED-FM", "DinoV2")
#MODEL = "STED-FM"
MODEL = "DinoV2"

#### Download and transform data

`DATASETS_PATH` can be changed to wherever the user wans the datasets to be downloaded to.

In [3]:
# Decide where to save datasets
DATASETS_PATH = "stedfm_datasets"
os.makedirs(DATASETS_PATH, exist_ok=True)

DATA_DIR = os.path.join(DATASETS_PATH, f"{DATASET_NAME}_dataset")
os.makedirs(DATA_DIR, exist_ok=True)

In [4]:
from etl import download_dataset, transform_dataset

# Download and transform dataset to fit PhenoMe's required structure
download_dataset(DATASET_NAME, DATA_DIR)
transform_dataset(DATASET_NAME, DATA_DIR, DATA_DIR)

Done!
Transforming Zooniverse dataset...


100%|██████████| 1357/1357 [00:00<00:00, 3947.47it/s]


Done!


### 1. Wrap the STED-FM pretrained model

In [5]:
class STEDFMWrapper(ModelWrapper):
    def __init__(self, device):
        # Load the pretrained STED-FM model
        model, cfg = get_pretrained_model_v2(
            name = "mae-lightning-small",
            weights = "MAE_SMALL_STED", 
            in_channels=1,
            as_classifier=True,
        )

        # Initialize the parent class which moves the model to the device and sets it to eval()
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        out = self.model.forward_features(tensor)
        return out

if MODEL == "STED-FM":
    wrapper = STEDFMWrapper(device=device)
elif MODEL == "DinoV2":
    wrapper = load_dinov2_model(model_name="dinov2_vits14_reg", device=device)

Using cache found in /Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


### 2. Use custom transforms

The default PhenoMe transforms apply standardisation based on ImageNet values, which isn't appropriate for the STED-FM model.

In [ ]:
import torchvision.transforms as T

TARGET_SIZE = 224

if MODEL == "STED-FM":
    # Transforms for STED-FM model
    custom_transforms = T.Compose([
        T.ToTensor(),
        T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
    ])
elif MODEL == "DinoV2":
    # Transforms for DinoV2 model
    custom_transforms = T.Compose([
        T.ToTensor(),
        T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])

### 3. Load data (find_files)

We assume CSV-based metadata. The STED datasets were already normalized. If raw data is used, it should be normalized based on percentile values from **entire images**, not by crop.

In [7]:

from phenome import make_dataframe_metadata_fn
import pandas as pd

images_dir = os.path.join(DATA_DIR, "images")
masks_dir = os.path.join(DATA_DIR, "masks") # Optional
metadata_dir = os.path.join(DATA_DIR, "metadata.csv")

metadata_df = pd.read_csv(metadata_dir)
metadata_fn = make_dataframe_metadata_fn(metadata_df)
file_df = pheno.find_files(images_dir, mask_dir=masks_dir, metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")

Found 1084 files total.
Matched masks: 1084/1084 images (0 missing).
DataFrame: 1084 files, 33 columns.


Found 1084 images


### 4. Process images

In [8]:
# Base path to outputs: checkpoints and plots
OUTPUT_PATH = "output"
os.makedirs(OUTPUT_PATH, exist_ok=True)

CHECKPOINTS_PATH = os.path.join(OUTPUT_PATH, "checkpoints")
os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

In [9]:
checkpoint_path = os.path.join(CHECKPOINTS_PATH, DATASET_NAME + f"_{MODEL}" + "_checkpoint.hdf5")

if MODEL == "STED-FM":
    pheno.process_images(wrapper, 
                        force_rgb=False,  # STED-FM model expects a single channel
                        checkpoint_path=checkpoint_path,
                        custom_transformations=custom_transforms,
                        )
elif MODEL == "DinoV2":
    pheno.process_images(wrapper, 
                    force_rgb=True,
                    channel_mode='combined',
                    checkpoint_path=checkpoint_path,
                    custom_transformations=custom_transforms,
                    )
pheno.compute_properties(property_preset="complete", checkpoint_path=checkpoint_path)

Processing batches:   0%|          | 0/34 [00:00<?, ?it/s]

Pipeline ready: 1084 images available (lazy loading from output/checkpoints/Zooniverse_DinoV2_checkpoint.hdf5).
Masks: 1084/1084 images.


Computing properties:   0%|          | 0/1084 [00:00<?, ?img/s]

Checkpoint: saved 1084 properties (refreshed).


,image_index,image_path,image_name,eccentricity,ring_1_mean,texture_homogeneity,ring_2_mean,intensity_min_masked,intensity_mean_masked,area,...,intensity_entropy,major_axis_length,intensity_std,extent,intensity_max_masked,texture_contrast,circularity,intensity_max,texture_correlation,perimeter
0,0,stedfm_datasets/Zooniverse_dataset/images/R-Ho...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0.925953,0.339275,0.620878,0.519159,0.098976,0.458038,434,...,7.040886,45.120640,0.204721,0.344992,1.000000,767.112488,0.297782,1.000000,0.902193,135.332001
1,1,stedfm_datasets/Zooniverse_dataset/images/R-Ho...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0.948557,0.245279,0.563892,0.392226,0.098976,0.334749,380,...,6.498264,42.653957,0.155814,0.400000,0.703072,440.471710,0.386492,0.703072,0.898478,111.154327
2,2,stedfm_datasets/Zooniverse_dataset/images/R-Ho...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0.890856,0.487086,0.561250,0.674885,0.122867,0.578299,2153,...,7.529780,93.911560,0.276777,0.441189,1.000000,811.364014,0.251407,1.000000,0.937135,328.048767
3,3,stedfm_datasets/Zooniverse_dataset/images/R-Ho...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0.943404,0.331333,0.475545,0.570802,0.146758,0.546239,250,...,6.975996,31.900108,0.266069,0.480769,1.000000,1133.115112,0.622994,1.000000,0.911773,71.012192
4,4,stedfm_datasets/Zooniverse_dataset/images/R-Ho...,R-Homer_GARSTAR635_M-PSD95_GAMAlexa594_GluGly_...,0.968693,0.455584,0.577176,0.776251,0.160410,0.683352,270,...,6.760745,39.942474,0.319684,0.441176,1.000000,2131.103271,0.409614,1.000000,0.890132,91.012192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1079,1079,stedfm_datasets/Zooniverse_dataset/images/R-Ri...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,0.556693,0.136599,0.181099,0.205077,0.066465,0.191685,210,...,6.281836,17.978178,0.081715,0.823529,0.419940,196.952774,0.983528,0.419940,0.821496,51.798988
1080,1080,stedfm_datasets/Zooniverse_dataset/images/R-Ri...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,0.779905,0.456528,0.379459,0.705635,0.241692,0.640191,516,...,7.899512,33.082180,0.280216,0.613555,1.000000,1039.812622,0.757879,1.000000,0.935577,92.497475
1081,1081,stedfm_datasets/Zooniverse_dataset/images/R-Ri...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,0.378156,0.102011,0.272820,0.141102,0.048338,0.126479,295,...,5.695748,20.405333,0.043718,0.739348,0.208459,102.122849,0.852549,0.208459,0.780238,65.941124
1082,1082,stedfm_datasets/Zooniverse_dataset/images/R-Ri...,R-Rim_GARSTAR635-M-Bassoon_GAMAlexa594_0MgGlyB...,0.473148,0.062722,0.287986,0.096092,0.027190,0.084953,184,...,5.227383,16.331713,0.036701,0.721569,0.184290,60.307396,0.991782,0.184290,0.772092,48.284271


### 5. Explore results

In [10]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()

### Generate plots

In [11]:
PLOTS_PATH = os.path.join(OUTPUT_PATH, "plots")
os.makedirs(PLOTS_PATH, exist_ok=True)

if DATASET_NAME == "NAS":
    color_by = "condition"
    fig=pheno.plot_tsne(color_by=color_by, return_fig=True, render_mode="svg", exclude={"condition":"Block"})
    fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_noblock_scatterplot.svg"))
elif DATASET_NAME == "Optim":
    color_by = "label"
elif DATASET_NAME == "Zooniverse":
    color_by = "protein"

fig=pheno.plot_tsne(color_by=color_by, return_fig=True, render_mode="svg")
fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_scatterplot.svg"))